# Q2F | Predict Test Data

> F. Make predictions of the labels on the test data, using the trained model with chosen (best) hyperparameters. Summarize performance using the RMSE. Three teams with the lowest RMSE (who followed correctly all the previous steps) will receive bonus points.

**IMPORTANT**

The test data is used once: the models are loaded exactly as D and E saved them and scored on the test days. Nothing is refit, tuned or chosen after.

# Answer Summary

**Predictions on the test data**
- Both saved models predict every test day from the 11 raw attributes, with nothing fit on the test days.

**The trained models with the chosen hyperparameters**
- D's linear regression and E's polynomial regression, each loaded exactly as its notebook selected, trained on the training days and saved.

**RMSE**
- E's polynomial model misses by less than D's linear model: 1055.3 against 1081.2 rentals per day.
- Both scores come from autumn and winter days, a season neither model was validated on.

**Previous steps followed**
- The test data is used once: nothing is refit, tuned or chosen after the test days are scored, and neither score is used to pick a model.

---

# Imports

In [1]:
import joblib
import pandas as pd
from sklearn.metrics import mean_squared_error

import Q2.consts as Q2_CONSTS
from consts import TEST_SIZE

from utils import read_csv_to_df, SPLIT

pd.set_option('display.float_format', '{:.1f}'.format)  # rentals

# Fetch Data

In [2]:
df = read_csv_to_df(path=Q2_CONSTS.DATA_PATH, parse_dates=["dteday"])
df = df.rename(columns=Q2_CONSTS.SHORT_NAMES)
_, test_df = SPLIT.chronological_split(df=df, date_col="Date", test_size=TEST_SIZE)

ATTRIBUTES = ['Temp', 'FeelsLike', 'Humidity', 'Wind', 'Year', 'IsHoliday', 'IsWorkday', 'Season', 'Month', 'Weekday', 'Weather']
X_test, y_test = test_df[ATTRIBUTES], test_df[Q2_CONSTS.TARGET]

# Method

- **Models:** the two models D and E selected and saved, loaded from `Q2/MODELS` as they are: D's linear regression (`linear_regression.joblib`) and E's polynomial regression (`polynomial_regression.joblib`). Each is one pipeline that takes the 11 raw attributes and builds its own columns, so nothing is fit on the test days.
- **Test days:** the last 20% of days (2012-08-07 to 2012-12-31), all after the training days. D's and E's validation blocks ran from 2012-02-05 to 2012-08-06, so neither model was validated on autumn or winter days.
- **Metric:** RMSE in rentals per day on every test day, the metric D and E used.
- **Test data used once:** each model's RMSE is reported as it comes. Neither score is used to pick, change or refit a model.

# Load the Models

In [3]:
MODELS = {  # the models D and E selected and saved, as they are
	'D: linear regression': joblib.load(Q2_CONSTS.MODEL_OUTPUT_DIR / 'linear_regression.joblib'),
	'E: polynomial regression': joblib.load(Q2_CONSTS.MODEL_OUTPUT_DIR / 'polynomial_regression.joblib'),
}

# Test RMSE

In [4]:
PREDICTIONS = {name: model.predict(X_test) for name, model in MODELS.items()}

pd.DataFrame({'test rmse': {name: mean_squared_error(y_test, predicted, squared=False) for name, predicted in PREDICTIONS.items()}})

,test rmse
D: linear regression,1081.2
E: polynomial regression,1055.3


### Notes / Observations

- Both saved models predict every test day as D and E saved them, with nothing refit.
- E's polynomial model misses by less than D's linear model on the test days: 1055.3 against 1081.2 rentals per day.
- Both scores come from autumn and winter days, a season neither model was validated on (see Method), so they show how each model carries into that season.